# Fase 4: Capas Complementarias (POI)
**Tema**: 2. Centralidad y corredores críticos de la red vial
**Requisito de Rúbrica**: Al menos una capa complementaria integrada espacialmente al grafo.

Para el análisis de corredores críticos, es clave entender **qué atrae el flujo vehicular**. Por ello, descargaremos Puntos de Interés (POI) que son grandes atractores de tráfico:
- Centros de salud (`amenity=hospital`, `clinic`)
- Centros educativos (`amenity=school`, `university`)
- Comercio mayor (`shop=mall`, `supermarket`)

Integraremos espacialmente estos POI identificando el nodo de la red vehicular más cercano a cada uno de ellos y asignando a dicho nodo el conteo de atractores.

In [1]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np

graphml_path = "../data/miraflores_drive_proj.graphml"
G_proj = ox.load_graphml(graphml_path)

print(f"Grafo cargado: {G_proj.number_of_nodes()} nodos, {G_proj.number_of_edges()} aristas.")

Grafo cargado: 1369 nodos, 2678 aristas.


## Descarga de Atractores de Tráfico (POI)

In [2]:
place_name = "Miraflores, Provincia de Lima, Lima Metropolitana, Lima, Peru"
tags = {
    'amenity': ['hospital', 'clinic', 'school', 'university'],
    'shop': ['mall', 'supermarket']
}

print("Descargando POI desde OSM...")
gdf_poi = ox.features_from_place(place_name, tags=tags)
print(f"Se encontraron {len(gdf_poi)} POIs atractores de tráfico.")

# Limpiar geometría: Si hay polígonos, tomar su centroide
gdf_poi = gdf_poi.copy()
gdf_poi['geometry'] = gdf_poi.centroid

# Proyectar POIs al mismo CRS del grafo (EPSG:32718)
gdf_poi = gdf_poi.to_crs("EPSG:32718")

Descargando POI desde OSM...


Se encontraron 132 POIs atractores de tráfico.


C:\Users\jg153\AppData\Local\Temp\ipykernel_1612\2741183070.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_poi['geometry'] = gdf_poi.centroid


## Integración Espacial
Buscaremos el nodo de la red vehicular más cercano a cada POI y contaremos cuántos POIs caen en el área de influencia de cada intersección.

In [3]:
# Obtener coordenadas X, Y
x = gdf_poi.geometry.x
y = gdf_poi.geometry.y

# Encontrar los nodos más cercanos a cada POI
print("Integrando POIs con la red vial...")
nearest_nodes = ox.distance.nearest_nodes(G_proj, X=x, Y=y)
gdf_poi['nearest_node'] = nearest_nodes

# Contar POIs por nodo
poi_counts = gdf_poi['nearest_node'].value_counts().to_dict()

# Asignar atributo al grafo
for n, data in G_proj.nodes(data=True):
    data['poi_count'] = str(poi_counts.get(n, 0))

nodos_con_poi = sum(1 for n, d in G_proj.nodes(data=True) if int(d['poi_count']) > 0)
print(f"Integración completa. {nodos_con_poi} nodos de la red son el acceso más cercano a uno o más POIs atractores.")

Integrando POIs con la red vial...
Integración completa. 120 nodos de la red son el acceso más cercano a uno o más POIs atractores.


## Guardar Grafo Enriquecido

In [4]:
ox.save_graphml(G_proj, "../data/miraflores_drive_poi.graphml")
print("Grafo enriquecido con POIs guardado exitosamente.")

Grafo enriquecido con POIs guardado exitosamente.
